In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from itertools import chain

# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")

### View do painel e premissas de simulação

Esta view reúne as informações da Gold para alimentar o painel e simular o potencial de redução dos custos de atendimento.

Os custos médios são calculados a partir dos registros com custo disponível na Gold. Os percentuais abaixo são hipóteses adotadas pelo grupo para construir o cenário; não foram estimados por um teste de implantação:

- `taxa_auto = 70%`: parcela dos chamados humanos classificados como totalmente automatizáveis que migraria para autosserviço. Os 30% restantes manteriam o custo atual.
- `taxa_validacao = 50%`: parcela dos chamados humanos classificados como automação com validação humana que migraria para o fluxo proposto.
- Dentro dessa parcela migrada, consideramos 70% com custo médio de autosserviço e 30% com custo médio do atendimento inicial.
- `ganho_produtividade = 12%`: redução hipotética aplicada diretamente ao custo dos chamados classificados como atendimento humano.

A classificação dos motivos por possibilidade de automação também é uma hipótese da proposta. Sua viabilidade deve ser validada com as equipes responsáveis, considerando complexidade, segurança e necessidade de atendimento humano.

O ganho projetado corresponde à diferença entre o custo atual e o custo simulado. Chamados que já estão em autosserviço mantêm seu custo, e custos ausentes permanecem nulos.

Os valores representam potencial bruto no período analisado, sem descontar investimento de implantação ou manutenção. A conversão desse potencial em economia financeira depende da possibilidade de reduzir despesas ou aproveitar a capacidade liberada.

Antes de definir metas, propomos um piloto para medir migração, resolução, recontato, satisfação e custo. Esses resultados permitirão substituir as hipóteses pelos percentuais observados.

In [0]:
%sql
CREATE OR REPLACE VIEW vcredit_medallion.gold.vw_dashboard_chamados
COMMENT 'Contrato estável entre a fato_chamados (SKs) e o Dashboard Lakeview. Modelo de custo projetado baseado em activity-based costing: chamados humanos migram para automação com taxa de sucesso; chamados em autosserviço mantêm custo atual (não há ganho em "automatizar o que já é automatizado"). Todas as premissas expostas como colunas para auditoria e ajuste.' AS
WITH params AS (
  SELECT
    CAST(0.70 AS DOUBLE) AS taxa_auto, -- % de chamados humanos "Totalmente Automatizável" que migram
    CAST(0.50 AS DOUBLE) AS taxa_validacao, -- % de chamados humanos "Validação Humana" que migram
    CAST(0.70 AS DOUBLE) AS split_auto_validacao, -- dos migrados, % resolvidos 100% pelo bot
    CAST(0.30 AS DOUBLE) AS split_n1_validacao, -- dos migrados, % que ainda precisam de N1
    CAST(0.12 AS DOUBLE) AS ganho_produtividade -- % de redução em chamados humanos que ficam humanos
),
custos_medios AS (
  -- Custo médio por bucket, medido nos dados. NULL é ignorado pelo AVG.
  SELECT
    AVG(
      CASE
        WHEN cn.tipo_canal = 'Autosservico' THEN f.custo_total
      END
    ) AS custo_medio_auto,
    AVG(
      CASE
        WHEN cn.canal = 'ATENDIMENTO INICIAL' THEN f.custo_total
      END
    ) AS custo_medio_n1,
    AVG(
      CASE
        WHEN cn.canal = 'ATENDIMENTO ESPECIALIZADO' THEN f.custo_total
      END
    ) AS custo_medio_n2
  FROM
    vcredit_medallion.gold.fato_chamados f
      JOIN vcredit_medallion.gold.dim_canal cn
        ON f.sk_canal = cn.sk_canal
  WHERE
    f.custo_total IS NOT NULL
),
base AS (
  SELECT
    -- ============ Chaves ============
    f.id_chamado,
    c.id_cliente,
    a.id_atendente,
    m.id_motivo,
    cn.id_canal,
    f.sk_cliente,
    f.sk_atendente,
    f.sk_motivo,
    f.sk_canal,
    f.sk_data,
    -- ============ DATA ============
    d.data AS data_abertura,
    d.ano,
    d.mes,
    d.nome_mes,
    d.ano_mes,
    d.trimestre,
    d.dia_semana,
    d.nome_dia_semana,
    d.fim_de_semana,
    f.hora_do_dia,
    f.hora_abertura_chamado,
    -- ============ CLIENTE ============
    c.regiao,
    c.idade,
    c.data_cadastro,
    c.dia_vencimento_fatura,
    -- ============ ATENDENTE ============
    a.nome_atendente,
    a.nivel_atendimento,
    a.status_atendente,
    -- ============ MOTIVO ============
    m.motivo,
    m.categoria,
    m.criticidade,
    -- ============ CANAL ============
    cn.canal,
    cn.tipo_canal,
    -- ============ FATO ============
    f.resolvido,
    f.tempo_espera_segundos,
    f.tempo_atendimento_segundos,
    f.tempo_total_segundos,
    f.ordem_temporal_invalida,
    f.custo_total,
    f.qtd_lancamentos_custo,
    f.qtd_custos_negativos,
    f.nota_media,
    f.qtd_pesquisas,
    -- ============ faixa_espera ============
    CASE
      WHEN f.tempo_espera_segundos IS NULL THEN 'Sem informação'
      WHEN f.tempo_espera_segundos <= 30 THEN '0-30s'
      WHEN f.tempo_espera_segundos <= 60 THEN '31-60s'
      WHEN f.tempo_espera_segundos <= 120 THEN '61-120s'
      WHEN f.tempo_espera_segundos <= 300 THEN '121-300s'
      ELSE '>300s'
    END AS faixa_espera,
    -- ============ nivel_automacao ============
    CASE
      m.id_motivo
      WHEN 5 THEN 'Totalmente Automatizável'
      WHEN 10 THEN 'Totalmente Automatizável'
      WHEN 8 THEN 'Totalmente Automatizável'
      WHEN 13 THEN 'Totalmente Automatizável'
      WHEN 6 THEN 'Totalmente Automatizável'
      WHEN 11 THEN 'Totalmente Automatizável'
      WHEN 4 THEN 'Automação com Validação Humana'
      WHEN 12 THEN 'Automação com Validação Humana'
      WHEN 3 THEN 'Automação com Validação Humana'
      WHEN 7 THEN 'Automação com Validação Humana'
      WHEN 16 THEN 'Automação com Validação Humana' -- cancelamento movido
      WHEN 9 THEN 'Requer Atendimento Humano'
      WHEN 15 THEN 'Requer Atendimento Humano'
      WHEN 14 THEN 'Requer Atendimento Humano'
      ELSE 'Não classificado'
    END AS nivel_automacao
  FROM
    vcredit_medallion.gold.fato_chamados f
      LEFT JOIN vcredit_medallion.gold.dim_cliente c
        ON f.sk_cliente = c.sk_cliente
      LEFT JOIN vcredit_medallion.gold.dim_atendente a
        ON f.sk_atendente = a.sk_atendente
      LEFT JOIN vcredit_medallion.gold.dim_motivo m
        ON f.sk_motivo = m.sk_motivo
      LEFT JOIN vcredit_medallion.gold.dim_canal cn
        ON f.sk_canal = cn.sk_canal
      LEFT JOIN vcredit_medallion.gold.dim_data d
        ON f.sk_data = d.sk_data
),
calc AS (
  SELECT
    b.*,
    p.taxa_auto,
    p.taxa_validacao,
    p.split_auto_validacao,
    p.split_n1_validacao,
    p.ganho_produtividade,
    cm.custo_medio_auto,
    cm.custo_medio_n1,
    cm.custo_medio_n2,
    CAST(
      p.split_auto_validacao * cm.custo_medio_auto
        + p.split_n1_validacao * cm.custo_medio_n1 AS DECIMAL(18, 2)
    ) AS custo_medio_validacao,
    -- Rótulo do gatilho que define a projeção desta linha
    CASE
      WHEN b.custo_total IS NULL THEN 'Sem Custo Registrado'
      WHEN b.tipo_canal = 'Autosservico' THEN 'Já em Autosserviço'
      WHEN b.nivel_automacao = 'Totalmente Automatizável' THEN 'Migração p/ Automação Total'
      WHEN b.nivel_automacao = 'Automação com Validação Humana' THEN 'Migração p/ Validação Humana'
      WHEN b.nivel_automacao = 'Requer Atendimento Humano' THEN 'Ganho de Produtividade Humano'
      ELSE 'Não Classificado'
    END AS gatilho_economia
  FROM
    base b CROSS JOIN params p CROSS JOIN custos_medios cm
)
SELECT
  c.*,
  -- ============ Aliases de compatibilidade (Lakeview) ============
  c.custo_total AS soma_custo,
  c.nota_media AS nota_media_avg,
  -- ============ Custo projetado (por regra) ============
  CASE
    c.gatilho_economia
    WHEN 'Sem Custo Registrado' THEN CAST(NULL AS DECIMAL(18, 2))
    WHEN 'Já em Autosserviço' THEN c.custo_total
    WHEN
      'Migração p/ Automação Total'
    THEN
      CAST(c.taxa_auto * c.custo_medio_auto + (1 - c.taxa_auto) * c.custo_total AS DECIMAL(18, 2))
    WHEN
      'Migração p/ Validação Humana'
    THEN
      CAST(
        c.taxa_validacao * c.custo_medio_validacao
          + (1 - c.taxa_validacao) * c.custo_total AS DECIMAL(18, 2)
      )
    WHEN
      'Ganho de Produtividade Humano'
    THEN
      CAST((1 - c.ganho_produtividade) * c.custo_total AS DECIMAL(18, 2))
    ELSE c.custo_total
  END AS custo_projetado_auto,
  -- ============ Custo economizado (delta, por regra) ============
  CASE
    c.gatilho_economia
    WHEN 'Sem Custo Registrado' THEN CAST(NULL AS DECIMAL(18, 2))
    WHEN
      'Migração p/ Automação Total'
    THEN
      CAST(
        c.custo_total
          - (c.taxa_auto * c.custo_medio_auto + (1 - c.taxa_auto) * c.custo_total) AS DECIMAL(18, 2)
      )
    WHEN
      'Migração p/ Validação Humana'
    THEN
      CAST(
        c.custo_total
          - (
            c.taxa_validacao * c.custo_medio_validacao + (1 - c.taxa_validacao) * c.custo_total
          ) AS DECIMAL(18, 2)
      )
    WHEN
      'Ganho de Produtividade Humano'
    THEN
      CAST(c.custo_total * c.ganho_produtividade AS DECIMAL(18, 2))
    ELSE CAST(0 AS DECIMAL(18, 2))
  END AS custo_economizado
FROM
  calc c;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

fato = spark.table(f"{GOLD_SCHEMA}.fato_chamados")
view = spark.table(f"{GOLD_SCHEMA}.vw_dashboard_chamados")

# ---- Tabela de checks ----
checks = []
def add_check(desc, qtd, critico=True):
    status = "OK" if qtd == 0 else ("FAIL" if critico else "WARN")
    checks.append((desc, int(qtd), status))
    if critico and qtd != 0:
        raise Exception(f"❌ {desc}: {qtd}")

# 1) Fan-out
add_check("fan-out (View != Fato)", view.count() - fato.count())

# 2) Cobertura de chaves naturais
for col in ["id_cliente", "id_motivo", "id_canal", "data_abertura"]:
    add_check(f"{col} nulo", view.filter(F.col(col).isNull()).count())
add_check("nome_atendente nulo", view.filter(F.col("nome_atendente").isNull()).count())

# 3) Colunas que o Lakeview referencia
refs = [
    "faixa_espera","nivel_automacao","custo_projetado_auto","custo_economizado",
    "gatilho_economia","taxa_auto","taxa_validacao","ganho_produtividade",
    "custo_medio_auto","custo_medio_n1","custo_medio_n2","custo_medio_validacao",
    "soma_custo","nota_media_avg","id_atendente","id_cliente","id_motivo","id_canal",
    "data_abertura","hora_do_dia","tempo_espera_segundos",
    "tempo_atendimento_segundos","tempo_total_segundos","custo_total",
    "resolvido","categoria","motivo","nivel_atendimento","nome_atendente","canal",
]
faltando = [c for c in refs if c not in view.columns]
assert not faltando, f"Colunas ausentes: {faltando}"

# 4) Tipo de data_abertura
assert dict(view.dtypes)["data_abertura"] == "date"

# 5) NULL preservado
add_check("NULL em custo_projetado_auto (deve ser igual ao da fato)",
          view.filter(F.col("custo_projetado_auto").isNull()).count()
          - fato.filter(F.col("custo_total").isNull()).count())

# 6) KPIs de volume idênticos
add_check("contagem divergente", view.count() - fato.count())

# 7) URA e Chatbot não ficam mais caros (modelo respeita autosserviço)
nao_autosservico = view.filter(F.col("tipo_canal") == "Autosservico")
add_check("autosserviço com custo projetado > atual (não deveria ocorrer)",
          nao_autosservico.filter(F.col("custo_projetado_auto") > F.col("custo_total")).count())

checks_df = spark.createDataFrame(checks, "check string, count long, status string")
display(checks_df)